[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [NumPy, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/numpy-deep-dive.html)

# The axis Argument &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
import math
from datetime import datetime, timedelta

import numpy as np

STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
START = datetime(2025, 1, 1)
HOURS = 365 * 24


def year_of_readings(station):
    """A reading for every hour of 2025 at one station, as the sqlite3 guide makes them.
    Svalbard was silent on 2 March, so those 24 hours are None."""
    i = list(STATIONS).index(station)
    row = []
    for n in range(HOURS):
        hour = START + timedelta(hours=n)
        if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
            row.append(None)
        else:
            season = -math.cos(2 * math.pi * (n - 400) / HOURS)
            day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
            wobble = ((n * 37 + i * 101) % 17 - 8) / 10
            row.append(round(STATIONS[station] + 9 * season + 3 * day + wobble, 1) + 0.0)
    return row


readings_list = [year_of_readings(station) for station in STATIONS]
readings = np.array(readings_list, dtype=float)       # None becomes nan


def hour_of_year(month, day, hour=0):
    """The position along a station's row of an hour of 2025. hour_of_year(7, 1, 12) is noon on 1 July."""
    return (datetime(2025, month, day, hour) - START) // timedelta(hours=1)


print("numpy", np.__version__, "|", len(readings_list), "stations,", len(readings_list[0]), "hours each |", readings.shape)

numpy 2.1.3 | 4 stations, 8760 hours each | (4, 8760)


**1.** Print each station's largest reading with `np.nanmax` and the axis that keeps the stations, and the shape of the result.

In [2]:
largest = np.nanmax(readings, axis=1)

print(largest, largest.shape)

[20.8 19.3  8.3 16.2] (4,)


Keeping the stations means combining the hours, which are axis 1.

**2.** Count, for every hour, how many stations were above 10 degrees, and print how many hours had at least three.

In [3]:
stations_above = (readings > 10).sum(axis=0)

print((stations_above >= 3).sum())

1847


Svalbard was never above 10 degrees, so three is the most any hour can have.

**3.** Print each station's coldest daily mean, averaging `by_day` over the hours with `axis=2` and then taking `np.nanmin` along the days.

In [4]:
by_day = readings.reshape(4, 365, 24)
daily = by_day.mean(axis=2)

print(daily.shape)
print(np.nanmin(daily, axis=1).round(2))

(4, 365)
[ -1.04  -2.54 -13.55  -5.55]


The first reduction leaves (4, 365), so the days are axis 1 of the result. Svalbard's 2 March is `nan`, which `np.nanmin` leaves out.

**4.** Use a tuple of axes on `by_day` to find the network's mean at each hour of the day, all four stations and all 365 days together, and print the values for 03:00 and 15:00, rounded to two places.

In [5]:
by_day = readings.reshape(4, 365, 24)
hour_of_day = np.nanmean(by_day, axis=(0, 1))

print(hour_of_day.shape)
print(hour_of_day[[3, 15]].round(2))

(24,)
[0.38 6.39]


Combining the stations and the days leaves the 24 hours of the day.

**5.** Subtract each day's mean from its hours with `keepdims=True`, and print the largest value in Tromso's result, row 3, rounded to two places.

In [6]:
by_day = readings.reshape(4, 365, 24)
within_day = by_day - by_day.mean(axis=2, keepdims=True)

print(within_day[3].max().round(2))

3.88


Tromso has no missing hours, so `max` needs no `nan` version.

---

&#8592; **Back to:** [The axis Argument](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/numpy-deep-dive/11-the-axis-argument.ipynb)  &nbsp;&middot;&nbsp;  [NumPy, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/numpy-deep-dive.html)
